# Entrega 05: vazamento, validação e resíduos

Aqui eu confiro se o modelo é avaliado em condições que imitam o uso real e analiso onde os erros se concentram. Os dados são sintéticos (mock), então nada daqui descreve o RU real.

In [9]:
import json
from pathlib import Path
import pandas as pd

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados").is_dir())
base = json.loads((raiz / "notebooks" / "E04_equipe02_dados_baseline.ipynb").read_text(encoding="utf-8"))

# Roda só as células do notebook coletivo até os baselines (seção 6), sem as análises avulsas.
for celula in base["cells"]:
    if celula["cell_type"] == "markdown" and "".join(celula["source"]).startswith("## 7"):
        break
    if celula["cell_type"] == "code":
        exec("".join(celula["source"]), globals())

Linhas brutas: {'catraca': 224880, 'estudantes': 2016, 'componentes': 5922}


,problema,linhas
0,ID de evento vazio,0
1,ID de evento repetido,0
2,data inválida,0
3,"refeição, curso ou vínculo vazio",0
4,situação não reconhecida,0
5,estudante/período/curso vazio,0
6,estudante/período repetido,0
7,chave de componente vazia,0
8,ID de componente repetido,0
9,evento sem vínculo cadastrado,0


Campos com valores ausentes:


'Nenhum no mock atual'

Tipos após leitura e conversão da data:


,catraca,estudantes,componentes
datetime64[us],1,0,0
string,13,18,18


... (9 outputs left out)

Erro por curso e refeição:


,baseline,curso,refeicao,linhas,MAE,RMSE
0,Média do treino,CURSO_TECNICO_A,almoco,313,3.96,4.42
1,Média do treino,CURSO_TECNICO_A,cafe_da_manha,313,2.09,2.45
2,Média do treino,CURSO_TECNICO_A,jantar,313,1.77,2.07
3,Média do treino,CURSO_TECNICO_B,almoco,313,4.11,4.55
4,Média do treino,CURSO_TECNICO_B,cafe_da_manha,313,2.14,2.61
5,Média do treino,CURSO_TECNICO_B,jantar,313,2.10,2.48
6,Média do treino,CURSO_TECNICO_C,almoco,313,3.96,4.42
7,Média do treino,CURSO_TECNICO_C,cafe_da_manha,313,2.16,2.59
8,Média do treino,CURSO_TECNICO_C,jantar,313,1.80,2.15
9,Média do treino,CURSO_TECNICO_D,almoco,313,4.01,4.51


Erro do total diário:


,baseline,linhas,MAE,RMSE
0,Média do treino,313,42.91,45.10
1,D-7 proposto,313,15.04,27.19


## 1. Vazamento nas entradas

O modelo usa quatro colunas: dia da semana, mês, tipo de refeição e curso. Todas são conhecidas na véspera. O alvo (`quantidade_servida`) não está nas entradas.

In [10]:
print("Entradas:", X_treino.columns.tolist())
print("Alvo nas entradas?", "quantidade_servida" in X_treino.columns)
print(treino["aberto"].value_counts().to_dict(), treino["extracao_completa"].value_counts().to_dict())
print("Linhas fora do modelo:", len(base_curso) - len(modelavel), "| domingos x combinações:", 410 * 18)

Entradas: ['dia_semana', 'mes', 'tipo_refeicao', 'curso']
Alvo nas entradas? False
{True: 33822} {True: 33822}
Linhas fora do modelo: 7380 | domingos x combinações: 7380


## 2. Partição e comparação justa

Treino até 31/12/2024, validação em 2025 e teste em 2026. Os dois baselines usam as mesmas linhas da validação, com MAE e RMSE.

In [11]:
print("Treino / validação / teste:", len(treino), len(validacao), len(teste))
print("Ordem das datas ok:", treino.data.max() < validacao.data.min() < teste.data.min())
print("Linhas da validação com os dois baselines:", len(comparaveis), "de", len(validacao))
display(metricas.loc[metricas.curso.eq("TODOS")])

Treino / validação / teste: 33822 5634 4914
Ordem das datas ok: True
Linhas da validação com os dois baselines: 5634 de 5634


,baseline,curso,refeicao,linhas,MAE,RMSE
18,Média do treino,TODOS,TODAS,5634,2.67,3.24
37,D-7 proposto,TODOS,TODAS,5634,2.14,2.95


## 3. Resíduos dos baselines

Resíduo = real menos previsto, na validação. Valor positivo quer dizer que a previsão ficou abaixo do real.

In [12]:
r = avaliacao.copy()
for nome in ["previsao_media", "previsao_d7"]:
    r["res_" + nome] = r["quantidade_servida"] - r[nome].astype(float)
cols = ["res_previsao_media", "res_previsao_d7"]
cols_ver = ["data", "curso", "tipo_refeicao", "quantidade_servida", "previsao_d7"]

print(r[cols].describe().round(2))
print("\nMaiores subestimações do D-7:"); print(r.nlargest(5, "res_previsao_d7")[cols_ver])
print("\nMaiores superestimações do D-7:"); print(r.nsmallest(5, "res_previsao_d7")[cols_ver])
print("\nPor refeição:"); print(r.groupby("tipo_refeicao")[cols].agg(["mean", "std"]).round(2))
print("\nPor curso:"); print(r.groupby("curso")[cols].agg(["mean", "std"]).round(2))
print("\nPor mês (D-7):"); print(r.groupby(r["data"].dt.month)[["res_previsao_d7"]].mean().round(2))

       res_previsao_media  res_previsao_d7
count              5634.0           5634.0
mean                 0.28             0.01
std                  3.23             2.95
min                 -8.45            -15.0
25%                 -2.21             -2.0
50%                   0.4              0.0
75%                  2.63              2.0
max                 10.63             15.0

Maiores subestimações do D-7:
           data            curso tipo_refeicao  quantidade_servida  \
3335 2025-08-05  CURSO_TECNICO_F        almoco                  18   
3277 2025-08-01  CURSO_TECNICO_B        almoco                  14   
4247 2025-10-02  CURSO_TECNICO_F        jantar                  14   
4538 2025-10-22  CURSO_TECNICO_C        almoco                  14   
581  2025-02-07  CURSO_TECNICO_F        almoco                  13   

      previsao_d7  
3335            3  
3277            1  
4247            1  
4538            1  
581             1  

Maiores superestimações do D-7:
        

In [13]:
semanal = r.set_index("data")[cols].resample("W").mean().round(2)
print(semanal.to_string())

            res_previsao_media  res_previsao_d7
data                                           
2025-01-05               -3.26            -0.19
2025-01-12                -3.2            -0.02
2025-01-19                -3.3             -0.1
2025-01-26               -3.25             0.05
2025-02-02               -3.27            -0.02
2025-02-09                0.61             3.88
2025-02-16                0.32            -0.29
2025-02-23                0.71             0.39
2025-03-02                0.68            -0.03
2025-03-09                0.77             0.08
2025-03-16                0.57            -0.19
2025-03-23                0.66             0.09
2025-03-30                0.97             0.31
2025-04-06                0.92            -0.05
2025-04-13                0.59            -0.33
2025-04-20               -0.07            -0.66
2025-04-27                 0.9             0.96
2025-05-04                0.64            -0.26
2025-05-11                0.78          

In [14]:
ativos = estudantes[estudantes["situacao_vinculo_periodo"] == "ativo"]
print(ativos.groupby("periodo_letivo").size())

periodo_letivo
2019.1    120
2019.2    120
2020.1    120
2020.2    120
2021.1    120
2021.2    120
2022.1    120
2022.2    120
2023.1    120
2023.2    117
2024.1    117
2024.2    117
2025.1    117
2025.2    138
2026.1    144
2026.2    144
dtype: int64


### O que os resíduos mostram

- O D-7 não tem viés geral (resíduo médio 0,01) e erra menos que a média do treino, mas tem caudas maiores, de -15 a +15.
- Na volta às aulas (semanas de 09/02 e 10/08) ele subestima em cerca de 4 refeições, porque olha a semana anterior, que ainda era recesso.
- Na primeira semana do recesso (06/07 e 28/12) ele superestima, pelo mesmo motivo ao contrário.
- A média do treino subestima de forma constante de agosto a dezembro, o que aponta mudança de nível da demanda em 2025.
- O almoço tem o maior erro. Nenhum curso destoa.

Esses números vêm dos baselines. A análise do modelo entra quando ele estiver no notebook coletivo.

## 4. Limitações

- Os dados são sintéticos.
- O erro típico é de cerca de 3 refeições por curso, por refeição e por dia, alto perto de uma média de 5.
- O modelo atual não enxerga o início e o fim do período letivo. A fase do calendário é conhecida na véspera e poderia entrar.
- A demanda cresce com o tempo, e o teste tem média 14,2% acima do treino (conferido na auditoria da Entrega 4). Parte disso coincide do aumento de vínculos ativos: 117 até 2025.1, 138 em 2025.2 e 144 em 2026, conforme a contagem acima.
- O recesso do mock é fixo (janeiro e julho inteiros), então os erros nessas semanas refletem a regra do gerador, não o RU real.
- Variável sugerida para o modelo: número de vínculos ativos no período, que é conhecido na véspera (vem do cadastro) e explicaria o salto de agosto de 2025.